# Week 10 extension — NLL ablations across all experiments (notebook 10e)

This is the evaluation half of the Week 10 extension. It scores every
checkpoint produced by 10d (`ckpt_E*.ckpt` in this directory) against
the same **hard-gated NLL** metric used in 10c, and adds a critical new
diagnostic: an **oracle MLP** that maps each experiment's cond vector
directly to per-bin Gaussian residual parameters. The oracle's NLL is
an *upper bound* on what any model can extract from a given cond set:

- Diffusion ≪ oracle  →  architecture is the bottleneck (try FiLM, CFG, Fourier).
- Oracle ≈ classical  →  the cond set itself doesn't help; try a different
  cond group or stop running that variant.

This is what makes the ablation scientifically honest: without an
oracle, a flat NLL across experiments could mean *either* "more cond
doesn't help" *or* "the architecture can't extract the new cond's
information" — two completely different fixes.

**The test split is reserved for the PI.** Every cell in this notebook
filters to `split in {"train", "val"}`.


In [ ]:
# Standard setup.
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat


In [ ]:
# Bootstrap sys.path and locate artifacts.
import os, sys

for _p in [os.path.abspath(os.path.join(repo_path, "weeks", "week_10")),
           os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from conditioned_infrastructure import find_week10_artifacts
paths = find_week10_artifacts(extra_required=[
    "diffusion_windows_v2.parquet",
    "data/composite_sunspot_groups_peak_area.csv",
])

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    ExtendedConditionalDiffusionLightning,
    sample_conditional_extended,
    build_model,
    load_trained_experiment,
    block_cond_concat,
    k_run_combined,
)
from butterflAI_model import ButterflAIModel

classical  = ButterflAIModel(paths["classical_weights"])
windows_v2 = pd.read_parquet(paths["parquet_v2"])
# Test split is reserved for the PI.
windows_v2 = windows_v2.loc[windows_v2["split"].isin(["train", "val"])].reset_index(drop=True)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_WEEK10_DIR = paths["week10_dir"]
print(f"v2 parquet (train+val): {len(windows_v2)} rows")
print(f"checkpoints dir       : {_WEEK10_DIR}")
print(f"device                : {device}")
print(classical)

---
## The experiment menu (must match 10d)

The dict below mirrors the one in 10d. The eval loop looks up each
discovered `ckpt_E*.ckpt` by name in this dict to know which cond
groups to assemble, which arch to instantiate, and (for E6) whether to
sweep the guidance weight.

If you redefined any experiment in 10d, mirror the change here.


In [ ]:
# Experiment menu — keep in sync with 10d. Add new specs as you propose
# them in 10d so the score loop can look them up by name.
_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     20000,
    "lr":             1e-3,
    "batch_size":     64,
}
def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    "E0": _spec(),
    # Mirror your 10d entries here, e.g.:
    #   "E1": _spec(consumed_keys=["cond_base", "cond_opp"], groups=["base", "opp"]),
    #   "E2": _spec(arch="film"),
}

# Guidance values to sweep when scoring a CFG-trained checkpoint.
CFG_GUIDANCE_W = [1.0, 1.5, 2.0, 3.0]

# Discover trained checkpoints.
import glob
_ckpt_paths = sorted(glob.glob(os.path.join(_WEEK10_DIR, "ckpt_E*.ckpt")))
_discovered = [os.path.splitext(os.path.basename(p))[0].replace("ckpt_", "")
               for p in _ckpt_paths]
print(f"discovered checkpoints: {_discovered}")
for name in _discovered:
    assert name in EXPERIMENTS, f"ckpt_{name}.ckpt has no entry in EXPERIMENTS"

---
## Task 67 — Build per-window evaluation blocks

Re-window the raw CSV the same way 10c does (per-window, 6-monthly,
≥ 20 obs per window) and tag each window with its v2 parquet row's
*entire* cond superset — every group, normalized later per-experiment
using the corresponding checkpoint's `cond_<g>_means` / `cond_<g>_stds`
buffers. We work with **per-window blocks only** in 10e — that is the
granularity at which the diffusion model is native, and the granularity
where any improvement over Week 10 will be most visible.


In [ ]:
# Task 67 — per-window blocks tagged with their v2 cond superset.
#
# Goal: rebuild per-window evaluation blocks for every (cycle, hemisphere)
# in train+val, identical to the per-window construction in 10c, and tag
# each block with the *raw* (unnormalized) vector from each cond group in
# the v2 parquet. Downstream cells normalize per-experiment using each
# checkpoint's buffers via `block_cond_concat`.
#
# Reference: 10c uses a 6-monthly window with min 20 obs per window.
#
# Expected outputs:
#   GROUP_COLS  — dict mapping group name ("base", "cyclehemi", "opp",
#                 "traj") → list of column names in windows_v2.
#                 Resolve "traj" dynamically from columns starting with
#                 "area_lag".
#   hemicycles  — list of dicts, one per hemicycle, carrying a "blocks"
#                 list. Each block dict must include:
#                   "center_decimal", "tau", "lats" (np.ndarray of |lat|),
#                   "groups_raw" (dict group → raw float32 vector).
#                 Each hemicycle dict must include "cycle", "hemisphere",
#                 "amplitude", "t0", "split", "blocks".

raw_df = pd.read_csv(paths["raw_csv"])
# TODO: parse dates, compute abs_lat, derive hemisphere, drop missing CYCLE.

# TODO: define GROUP_COLS (see ExtendedConditionalResidualDataset.GROUP_COLS
#       in conditioned_infrastructure.py for the schema).

# TODO: build a per-window cond lookup keyed by (cycle, hemisphere, year_center).

# TODO: build_per_window_hc(cyc, hemi, df_hc) — returns the block list for
#       one hemicycle (mirror 10c's window construction).

hemicycles = []  # TODO: populate
assert len(hemicycles) > 0, "rebuild the per-window blocks before continuing"
assert all("groups_raw" in blk for hc in hemicycles for blk in hc["blocks"]), \
    "every block needs a groups_raw dict"
print(f"per-window blocks built: {sum(len(hc['blocks']) for hc in hemicycles)}")
print(f"hemicycles included    : {len(hemicycles)}")

---
## Task 67 (cont) — NLL primitives, same as 10c

These are byte-identical to the primitives in 10c. Re-stated here so
10e can be run standalone (without executing 10c first).


In [ ]:
# Task 67 — hard NLL primitives. Port from 10c (or import them if you've
# factored them out). The two callables you need are:
#
#   hard_nll_classical(model, hcs) -> (nll, detail)
#   hard_nll_combined (model, hcs, residuals_by_block, eps=1e-6)
#                                                  -> (nll, detail)
#
# 10c defines both; they are byte-identical here. Once defined, the
# print below should report a classical baseline near 10c's val number.

# TODO: define hard_nll_classical and hard_nll_combined.

val_hcs = [hc for hc in hemicycles if hc["split"] == "val"]
nll_cl_val, det_cl_val = hard_nll_classical(classical, val_hcs)
print(f"classical hard NLL (val): {nll_cl_val:.4f}  "
      f"(coverage {det_cl_val['coverage']:.3f})")

---
## Task 67 (cont) — Diagnostic oracle

For each experiment's cond set, fit a tiny MLP that maps the cond
vector directly to a 15-D Gaussian over the residual bins
(`mean`, `log_std`). The oracle's NLL is computed by sampling K
residuals from the per-block Gaussian and feeding them through
`hard_nll_combined` — the same harness used to score the diffusion.

What the oracle answers:

- **Diffusion ≪ oracle**  →  the diffusion isn't extracting the
  information that's already in the cond. Try a stronger architecture
  (FiLM, Fourier features, larger MLP).
- **Oracle ≈ classical**  →  the cond set itself doesn't carry enough
  information about the residual structure. Try a different cond
  group or stop adding to this one.

You implement this. The science (the tiny MLP, the Gaussian NLL
expression, the training loop, sampling from the predicted Gaussian)
is yours.

In [ ]:
# Task 67 — the oracle MLP. Students fill in the bodies below.

class OracleMLP(nn.Module):
    """Map a cond vector to per-bin Gaussian residual params (mean, log_std).

    Output is 2 * 15 = 30 numbers per row: 15 means + 15 log-stds.
    """
    def __init__(self, cond_dim, hidden_dim=64):
        super().__init__()
        # TODO: build a 2–3 layer MLP with SiLU activations.
        raise NotImplementedError("Task 67 — implement OracleMLP.__init__")

    def forward(self, cond):
        # TODO: return (mean, log_std), each shape (B, 15).
        raise NotImplementedError("Task 67 — implement OracleMLP.forward")


def gaussian_nll(r, mean, log_std):
    """Per-row, per-bin Gaussian NLL of the *standardized* residual ``r``
    under the predicted ``(mean, log_std)``. Return a scalar."""
    # TODO: implement the closed-form Gaussian NLL.
    raise NotImplementedError("Task 67 — implement gaussian_nll")


def fit_oracle(cond_train, r_train, cond_val, r_val,
               max_epochs=500, lr=1e-2, hidden_dim=64, seed=0):
    """Fit OracleMLP on (cond_train, r_train); track val NLL each epoch
    and return the module with the best val state restored, along with
    the best val NLL. ``r_*`` are *standardized* residuals (15-D)."""
    # TODO: instantiate OracleMLP, an Adam optimizer, and a training
    #       loop with early stopping on best val gaussian_nll.
    raise NotImplementedError("Task 67 — implement fit_oracle")

---
## Task 68 — Score every checkpoint

For each discovered `ckpt_E*.ckpt`:

1. Build per-experiment cond tensors for every val block by
   concatenating the right groups in `consumed_keys` order, normalized
   with the **checkpoint's own** per-group buffers (so val data uses
   train-set normalization recovered from the saved model).
2. Run K = 100 conditional samples per block using
   `sample_conditional_extended`. For E6 (CFG), repeat the sampling at
   every guidance weight in `CFG_GUIDANCE_W` and keep them as separate
   rows.
3. Fit the oracle MLP on the same (cond, standardized residual) data
   and record its val NLL as the upper bound for this cond set.
4. Plug each of the K samples into `hard_nll_combined`; report mean
   and σ over K.


In [ ]:
# Task 68 — score every discovered checkpoint.

K = 100   # K samples per val block; bump to 500 for tighter K-σ bars.


def score_checkpoint(name, cfg):
    """Score one experiment.

    The recipe:
      1. Load the trained model + datasets via
         `load_trained_experiment(name, cfg, windows_v2, _WEEK10_DIR,
                                  alpha_np, sigma_np)`.
      2. Build per-val-block cond tensors via
         `block_cond_concat(val_hcs, lit, cfg, train_ds)`,
         repeat each row K times, and sample residuals with
         `sample_conditional_extended(lit, cond_K, guidance_w=w,
                                       device=device)`.
         If `cfg["cond_dropout_p"] > 0`, sweep every `w` in
         `CFG_GUIDANCE_W` and emit one row per `w`; otherwise sample
         once at `w = 0.0`.
      3. Reshape samples to (N, K, 15) physical units; push them through
         `k_run_combined(hard_nll_combined, classical, val_hcs, keys,
                          samples_NK15)` to get K NLLs and floor
         fractions.
      4. Fit the oracle on the same cond set: build (cond, residual)
         pairs for train and val blocks, standardize residuals with the
         train-set bin stats, and call `fit_oracle`. Convert the
         oracle's per-block Gaussian into K physical-unit samples and
         push them through `k_run_combined` to get the oracle's
         hard-NLL upper bound.

    Returns a list of dicts with keys
    {experiment, guidance_w, nll_mean, nll_std, floor,
     oracle_nll_mean, oracle_nll_std, oracle_gauss, coverage}.
    """
    # TODO: implement the four steps above.
    raise NotImplementedError("Task 68 — implement score_checkpoint")


all_rows = []
for name in _discovered:
    print(f"scoring {name} ...")
    all_rows.extend(score_checkpoint(name, EXPERIMENTS[name]))

scoreboard = pd.DataFrame(all_rows)
scoreboard["classical"] = nll_cl_val
print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

---
## Task 69 — Headline plot

Bar chart, val split: classical baseline plus every experiment's NLL,
with K-σ error bars. Oracle NLL per experiment overlaid as a
horizontal dashed marker to make the "what's achievable from this cond
set" boundary visible.

For any CFG variant (`cond_dropout_p > 0`), the bar shown is the
best-NLL guidance setting; a secondary panel sweeps the guidance
weight `w` so you can see the guidance vs NLL trade.

In [ ]:
# Task 69 — headline plot + CFG sweep + per-hemicycle breakdown.
#
# Required panels:
#   1. Bar chart of val NLL: classical bar on the left, one bar per
#      experiment (best guidance w if it's a CFG variant), with K-σ
#      error bars. Overlay each experiment's oracle NLL as a dashed
#      horizontal marker so the "what's achievable from this cond set"
#      boundary is visible.
#   2. (If any CFG checkpoint exists) a sweep of guidance weight vs NLL
#      on the CFG variant.
#   3. Per-hemicycle breakdown for the best variant — same axes as the
#      Week 10 chart so any improvement is visually unambiguous.
#
# Useful values you already have:
#   - `scoreboard` (DataFrame from Task 68)
#   - `nll_cl_val` (classical baseline)
#   - `val_hcs`, `EXPERIMENTS`, `K`, `_WEEK10_DIR`
#
# For panel 3, reuse `load_trained_experiment(...)`,
# `block_cond_concat(...)`, `sample_conditional_extended(...)`, and
# `k_run_combined(...)`.

# TODO: build the three panels.

---
## Task 70 — Going further

Once you've worked through the Level 1–5 escalation menu in 10d and
want to push further, the tiered menu below ranks the next experiments
by expected payoff per unit effort. Discipline still applies: one knob
at a time, log to wandb, add to `EXPERIMENTS` in both 10d and 10e,
then re-run.

**Level 1 — easy wins**
- **Wider/deeper MLP.** Bump `hidden_dim` from 128 to 256, `n_layers`
  from 3 to 5 in the winning experiment's config. If NLL drops, the
  network was capacity-bound — interesting on its own.
- **Longer K at evaluation.** Bump K from 100 to 500 for the winning
  variant — tightens the K-σ error bar and lets you trust smaller
  margins.
- **Sampler comparison.** Re-score the winner with DDPM (stochastic)
  sampling instead of the deterministic DDIM in
  `sample_conditional_extended`. Deterministic samplers can under-
  disperse, inflating NLL.

**Level 2 — extra cond information**
- **Larger trajectory K.** Bump `K_LAGS` from 4 to 8 in 10d. If the
  trajectory variant's oracle improves but its diffusion doesn't, the
  architecture is underusing the longer history.
- **Lagged opposite-hemisphere.** Pair the trajectory cond with the
  opposite hemisphere — `opp_area_smoothed_lag1..lag4`.

**Level 3 — architectural changes**
- **Cross-attention conditioning.** Replace the FiLM mechanism with
  cross-attention over a small set of learned cond tokens — overkill
  for the cond dim here, but worth knowing if the FiLM gain saturates.
- **Per-bin-aware loss.** Weight the ε-prediction loss by the inverse
  per-bin std so well-resolved bins don't dominate gradients.

**The test set is the PI's.** Every iteration above is val-only. The
final test-set reveal happens once, after the program is closed.

---
## Handoff back to the PI

The headline numbers in `scoreboard` answer two questions:

1. **Does any variant beat the classical baseline on val?** If yes, the
   diffusion approach has earned its place in the final pipeline.
2. **Where is the bottleneck — information or architecture?** Compare
   each row's `nll_mean` to its `oracle_nll_mean`. A large gap means
   the cond set has more information than the diffusion is extracting
   (architecture-bound). A small gap with the oracle near classical
   means the cond set isn't carrying enough information — that line of
   experiments is exhausted; try a different cond group.

The PI will run the test set evaluation on whatever variant the val
results recommend.
